# 重回帰分析

重回帰は複数特徴量の寄与を同時に推定できる最小構成の予測モデルです。単回帰より現実的な設定で、解釈可能性と実装容易性を両立できます。


## 背景と目的

実データは複数要因が同時に効くため、単一説明変数だけで予測すると係数解釈が歪みます。

重回帰を理解すると、特徴量追加が予測と解釈に与える効果を定量的に評価できます。

行列表現・最小二乗解・多重共線性指標を順に確認します。


## 最初に解きたい疑問

1. `x1` と `x2` を同時に使うと、単回帰より何がうれしいのか。
2. `X = np.column_stack([np.ones(n), x1, x2])` の先頭の `1` の列は何のために入れているのか。
3. `beta = (X^T X)^(-1) X^T y` は何を求めている式なのか。
4. `VIF` が大きいと何がまずいのか。
5. 係数が真の式と少し違っても、`train MSE` が小さければ十分なのか。


## 先に押さえる言葉

- `重回帰`: 複数の入力特徴量を同時に使って予測する方法です。1つの要因だけでなく、複数の要因の影響をまとめて見られます。
- `切片`: 説明変数が0のときの基準値です。回帰式では、直線や平面の出発点になります。
- `多重共線性`: 入力どうしが強く似ていて、役割がかぶっている状態です。こうなると、係数が不安定になりやすくなります。
- `VIF`: 説明変数が他の説明変数でどれだけ説明できてしまうかを表す指標です。値が大きいほど、変数どうしのかぶりが強いと考えます。


## 実行前の見取り図

1. `beta_hat` が、データ生成式の係数 `[1.2, 2.0, -1.5]` にどれだけ近いかを見る。
2. `train MSE` が小さいかを確認する。
3. `VIF(x1)` と `VIF(x2)` が大きくなっていないかを確認する。


## 出力の読み方

- `beta_hat` の符号や大きさをどう読めばよいか。


## つまずきやすい点

- 行列の式が、どうやって係数の推定値に変わるのかを、各記号の役割ごとに説明する必要があります。
- `VIF` が高いと何が困るのかを、予測ではなく解釈の不安定さとして言い換える説明が必要です。
- X^T X が不安定なときに pinv を使う理由。
- VIF は解釈の不安定さを見る指標で、予測精度そのものではないこと。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- VIF の値だけで特徴量の採否を決められないこと。


## コード 1: 入力データを作る

このセルでは 入力データを作る ための最小コードを動かします。 実行時は「`beta_hat` が、データ生成式の係数 `[1.2, 2.0, -1.5]` にどれだけ近いかを見る。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
import numpy as np
np.random.seed(11)

n = 120
x1 = np.random.randn(n)
x2 = 0.6 * x1 + 0.8 * np.random.randn(n)  # 相関あり
eps = 0.5 * np.random.randn(n)
y = 1.2 + 2.0 * x1 - 1.5 * x2 + eps

X = np.column_stack([np.ones(n), x1, x2])


## コード 2: 実行環境をそろえる

このセルでは 実行環境をそろえる ための最小コードを動かします。 実行時は「`train MSE` が小さいかを確認する。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
# 最小二乗解: beta = (X^T X)^(-1) X^T y
beta = np.linalg.pinv(X.T @ X) @ X.T @ y
pred = X @ beta
mse = np.mean((pred - y) ** 2)

print('beta_hat =', np.round(beta, 6))
print('train MSE=', round(mse, 6))


## コード 3: 更新式や補助関数を定義する

このセルでは 更新式や補助関数を定義する ための最小コードを動かします。 実行時は「`VIF(x1)` と `VIF(x2)` が大きくなっていないかを確認する。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
# VIFで多重共線性を確認
# VIF_j = 1 / (1 - R_j^2)

def vif(x_target, x_other):
    Xo = np.column_stack([np.ones(len(x_other)), x_other])
    b = np.linalg.pinv(Xo.T @ Xo) @ Xo.T @ x_target
    pred = Xo @ b
    ss_res = np.sum((x_target - pred) ** 2)
    ss_tot = np.sum((x_target - x_target.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    return 1.0 / (1.0 - r2)

vif_x1 = vif(x1, x2)
vif_x2 = vif(x2, x1)
print('VIF(x1)=', round(vif_x1, 6), 'VIF(x2)=', round(vif_x2, 6))


係数推定値だけでなく VIF を併せて見ることで、予測精度と解釈安定性の両方を評価できます。
